# Day 60 - beam search and speculative decoding

Two decoding strategies that pull in opposite directions. Beam search changes the output
to find a more probable sentence and costs more. Speculative decoding changes nothing
about the output distribution and costs less.

The grammar here admits 1,250 sentences, so both claims can be checked against an
enumeration rather than eyeballed.

In [ ]:
import numpy as np
from decoding import (NGram, make_corpus, all_sentences, logprob, exact_ranking,
                      greedy, beam_search, beam_table, beam_monotonicity, length_bias,
                      beam_diversity, speculative_distribution,
                      always_accept_distribution, deterministic_accept_distribution,
                      speculative_generate, draft_table, gamma_table, is_grammatical,
                      where_the_beam_loses, width_needed, verify_in_one_pass,
                      tree_table, batch_model)

text = make_corpus()
m = NGram(text)
rank = exact_ranking(m)
print(len(rank), 'sentences scored exactly')
print('best  ', repr(rank[0][1]), round(rank[0][0], 3))
g = greedy(m)
print('greedy', repr(g), 'rank', [s for _, s in rank].index(g) + 1)

1250 sentences scored exactly
best   'a small dog sat in the garden.' -6.244
greedy 'the red cat sang in the garden.' rank 44


## 1. Beam search

Keep the `width` best prefixes at every step instead of only the best one.

In [ ]:
for r in beam_table(m):
    print(f"width {r['width']:3d}  rank {str(r['rank']):>5}  log p {r['lp']:7.3f}  {r['text']!r}")

width   1  rank    44  log p  -6.717  'the red cat sang in the garden.'
width   2  rank    44  log p  -6.717  'the red cat sang in the garden.'
width   4  rank    10  log p  -6.502  'the quiet robot sang in the garden.'
width   8  rank     5  log p  -6.407  'the old robot sang in the garden.'
width  16  rank     5  log p  -6.407  'the old robot sang in the garden.'
width  32  rank     5  log p  -6.407  'the old robot sang in the garden.'


## 2. Where exactly the beam loses the best sentence

Follow the winning prefix character by character and watch the step where a width-8
beam drops it. Nothing recovers after that.

In [ ]:
wl = where_the_beam_loses(m, width=8)
print(wl)
for r in width_needed(m):
    print('width', r['width'], 'finds it' if r['found'] else 'misses it')

{'best': 'a small dog sat in the garden.', 'step': 9, 'prefix': 'a small d', 'rank_among_candidates': 10, 'candidates': 64, 'width': 8}
width 32 misses it
width 64 misses it
width 128 finds it


## 3. A wider beam is not always better

Beam search is a heuristic. On a model trained on a smaller corpus, widening the beam
can make the answer *less* likely - the effect translation papers call the beam search
curse.

In [ ]:
bm = beam_monotonicity(NGram(make_corpus(400, seed=0)))
for r in bm['rows'][:8]:
    print(f"width {r['width']:2d}  log p {r['lp']:7.3f}")
print('widths where widening made it worse:', len(bm['drops']))

width  1  log p  -6.188
width  2  log p  -6.188
width  3  log p  -6.422
width  4  log p  -6.224
width  5  log p  -6.224
width  6  log p  -6.141
width  7  log p  -5.947
width  8  log p  -5.947
widths where widening made it worse: 1


## 4. The length bias, and the diversity collapse

In [ ]:
lb = length_bias(m)
print('corr(length, log p) =', round(lb['corr'], 3))
print('mean length: all', lb['mean_len'], ' top-10 by total', lb['top_total_len'],
      ' top-10 per character', lb['top_char_len'])

bd = beam_diversity(m)
print('\nbeam   :', bd['beam_cov'])
print('sampled:', {k: round(v, 2) for k, v in bd['sample_cov'].items()})

corr(length, log p) = 0.061
mean length: all 30.6  top-10 by total 28.8  top-10 per character 36.5

beam   : {'n': 16, 'legal': 16, 'subjects': 8, 'overlap': 1.5333333333333334}
sampled: {'subjects': 12.75, 'overlap': 1.34, 'legal': 14.58, 'sets': 12}


## 5. Why verifying gamma tokens costs one pass

A causal model scores position i from the tokens to its left, so one pass over
`context + proposals` produces the distribution at every one of the gamma+1 positions -
the same computation as a training step.

In [ ]:
props = [m.stoi[c] for c in 'ang ']
one = verify_in_one_pass(m, '\nthe old robot s', props)
ctx = '\nthe old robot s'
for i, p_ in enumerate(one):
    assert np.abs(p_ - m.probs(ctx)).max() < 1e-15      # same as scoring one by one
    top = int(np.argmax(p_))
    print(f"position {i}: top {m.vocab[top]!r} at {p_[top]:.3f}")
    if i < len(props):
        ctx += m.vocab[props[i]]

position 0: top 'a' at 0.691
position 1: top 'n' at 0.499
position 2: top 'g' at 0.996
position 3: top ' ' at 0.996
position 4: top 'i' at 0.296


## 6. Speculative decoding is exact

Accept each draft token with probability `min(1, p/q)`; replace the first rejection with
a draw from the normalised positive part of `p - q`. Adding both routes gives back `p`.

In [ ]:
d2 = NGram(text, order=2)
for ctx in ('\nthe ', '\nthe small cat ', '\na red dog s'):
    p, q = m.probs(ctx), d2.probs(ctx)
    out, acc = speculative_distribution(p, q)
    a, _ = always_accept_distribution(p, q)
    b, _ = deterministic_accept_distribution(p, q)
    print(f"{ctx.strip()!r:18} accept {acc:.3f}  correct {np.abs(out - p).max():.1e}"
          f"  trust-draft {np.abs(a - p).max():.3f}  no-coin {np.abs(b - p).max():.3f}")

'the'              accept 0.503  correct 0.0e+00  trust-draft 0.131  no-coin 0.043
'the small cat'    accept 0.447  correct 1.1e-16  trust-draft 0.367  no-coin 0.001
'a red dog s'      accept 0.753  correct 1.1e-16  trust-draft 0.247  no-coin 0.000


## 7. The two knobs

How good the draft is, and how far ahead it guesses. Neither changes the output.

In [ ]:
for r in draft_table(m, text):
    print(f"order {r['order']}  accept {r['accept']:.3f}  tokens/round {r['per_round']:.2f}"
          f"  grammatical {r['grammatical']:.0%}")
print()
for r in gamma_table(m, NGram(text, order=3)):
    print(f"gamma {r['gamma']}  accept {r['accept']:.3f}  measured {r['per_round']:.2f}"
          f"  iid theory {r['theory']:.2f}  speedup {r['speedup']:.2f}x")

order 1  accept 0.173  tokens/round 1.69  grammatical 92%
order 2  accept 0.605  tokens/round 3.42  grammatical 92%
order 3  accept 0.837  tokens/round 4.35  grammatical 90%
order 5  accept 0.929  tokens/round 4.72  grammatical 88%

gamma 1  accept 0.911  measured 1.91  iid theory 1.91  speedup 1.74x
gamma 2  accept 0.910  measured 2.82  iid theory 2.74  speedup 2.35x
gamma 4  accept 0.842  measured 4.37  iid theory 3.65  speedup 3.12x
gamma 8  accept 0.760  measured 7.08  iid theory 3.82  speedup 3.94x


## 8. Tree drafting, and the batch size where speculation stops paying

In [ ]:
for r in tree_table(m, text):
    print(f"draft order {r['order']}: chain {r['chain']:.2f}  tree {r['tree']:.2f}"
          f"   per position chain {r['chain_per_node']:.2f} / tree {r['tree_per_node']:.2f}")
print()
for r in batch_model(4.41, 4):
    print(f"batch {r['batch']:4d}  plain {r['base_ms']:6.1f} ms  spec {r['spec_ms']:6.1f} ms"
          f"  speedup {r['speedup']:.2f}x")

draft order 1: chain 1.65  tree 1.96   per position chain 0.41 / tree 0.68
draft order 2: chain 3.54  tree 3.32   per position chain 0.88 / tree 0.91
draft order 3: chain 4.41  tree 3.72   per position chain 1.10 / tree 0.97

batch    1  plain   28.0 ms  spec   28.0 ms  speedup 4.41x
batch    8  plain   28.0 ms  spec   28.0 ms  speedup 4.41x
batch   32  plain   28.0 ms  spec   28.0 ms  speedup 4.41x
batch  128  plain   28.0 ms  spec   89.6 ms  speedup 1.38x
batch  512  plain   71.7 ms  spec  358.4 ms  speedup 0.88x


## 9. Tests

In [ ]:
assert beam_search(m, width=1)[1] == greedy(m)
assert max(beam_search(m, width=w)[0] for w in (1, 2, 4, 8)) <= rank[0][0] + 1e-9
p, q = m.probs('\nthe small cat '), NGram(text, order=2).probs('\nthe small cat ')
out, _ = speculative_distribution(p, q)
assert np.abs(out - p).max() < 1e-12
assert np.abs(always_accept_distribution(p, q)[0] - p).max() > 1e-3
print('all assertions passed')

all assertions passed
